# Assignment 6: Pizza Ordering Chatbot

**Course:** AAI 6620 - Natural Language Processing  
**Student:** Bandari Ruthvik Nath  
**Email:** bandari.ru@northeastern.edu  

---

This notebook implements a pizza ordering chatbot using the **ChatterBot** library (v1.2.13). The bot features:

- A **custom logic adapter** (`PizzaOrderLogicAdapter`) that handles the full pizza ordering workflow
- Ordering pizzas by **size**, **toppings**, or **specialty name**
- Collecting **delivery address** and **credit card payment** details
- A **"what do you know"** command to display all collected order information
- Warm, conversational tone with friendly responses
- General chitchat handled by `BestMatch` adapter with `ListTrainer` data

## 1. Installation & Setup

In [18]:
%pip install chatterbot==1.2.13
%pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 59.2 MB/s  0:00:00eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
# Environment check (self-contained notebook setup)
print('Environment ready. ChatBot uses SQLite in-memory storage (no database file written).')

Environment ready. ChatBot uses SQLite in-memory storage (no database file written).


## 2. Custom Logic Adapter: PizzaOrderLogicAdapter

ChatterBot requires custom logic adapters to be in a **separate `.py` file**. We use `%%writefile` to create the adapter module, which handles:

- **Intent detection** via keyword matching for pizza orders, sizes, toppings, addresses, payment, and summaries
- **State management** to track the ordering flow (idle -> ordering -> address -> payment -> confirmed)
- **Menu system** with 7 specialty pizzas, 3 sizes, and 16 available toppings
- **Order summary** to display everything the bot knows about the current order

In [35]:
%%writefile pizza_adapter.py
"""
PizzaOrderLogicAdapter - Custom ChatterBot logic adapter for pizza ordering.

Handles the complete pizza ordering workflow including:
- Pizza selection (specialty or custom with toppings)
- Size selection (small, medium, large)
- Delivery address collection
- Credit card payment collection
- Order summary display
"""

from chatterbot.logic import LogicAdapter
from chatterbot.conversation import Statement


# ──────────────────────────────────────────────────────────────────────
# Pizza Menu Configuration
# ──────────────────────────────────────────────────────────────────────

PIZZA_MENU = {
    'specialty_pizzas': {
        'margherita':   ['mozzarella', 'tomato sauce', 'fresh basil'],
        'pepperoni':    ['mozzarella', 'pepperoni', 'tomato sauce'],
        'hawaiian':     ['mozzarella', 'ham', 'pineapple', 'tomato sauce'],
        'veggie':       ['mozzarella', 'bell peppers', 'mushrooms', 'onions', 'olives', 'tomato sauce'],
        'meat lovers':  ['mozzarella', 'pepperoni', 'sausage', 'bacon', 'ham', 'tomato sauce'],
        'bbq chicken':  ['mozzarella', 'grilled chicken', 'red onions', 'bbq sauce'],
        'supreme':      ['mozzarella', 'pepperoni', 'sausage', 'bell peppers', 'mushrooms', 'onions', 'olives'],
    },
    'sizes': ['small', 'medium', 'large'],
    'prices': {'small': 8.99, 'medium': 11.99, 'large': 14.99},
    'topping_surcharge': 1.50,
    'available_toppings': [
        'pepperoni', 'sausage', 'mushrooms', 'onions', 'olives',
        'bell peppers', 'bacon', 'ham', 'pineapple', 'jalapenos',
        'extra cheese', 'tomatoes', 'grilled chicken', 'anchovies',
        'fresh basil', 'spinach',
    ],
}


# ──────────────────────────────────────────────────────────────────────
# Order State (module level, persists across adapter calls)
# ──────────────────────────────────────────────────────────────────────

def _fresh_order():
    """Return a blank order dictionary."""
    return {
        'stage': 'idle',
        'items': [],
        'current_pizza': {},
        'delivery_address': '',
        'payment': {},
    }

order = _fresh_order()


def reset_order():
    """Reset the order state (useful for demo reruns)."""
    global order
    order = _fresh_order()


# ──────────────────────────────────────────────────────────────────────
# Helper: build a readable order summary
# ──────────────────────────────────────────────────────────────────────

def _format_summary():
    """Return a friendly summary of everything we know about the order."""
    lines = []
    lines.append('Here is everything I have for your order so far:')
    lines.append('')

    if order['items']:
        total = 0.0
        for i, pizza in enumerate(order['items'], 1):
            name = pizza.get('name', 'Custom pizza')
            size = pizza.get('size', 'medium')
            toppings = pizza.get('toppings', [])
            base_price = PIZZA_MENU['prices'].get(size, 11.99)
            extra_count = max(0, len(toppings) - 3)
            price = base_price + (extra_count * PIZZA_MENU['topping_surcharge'])
            total += price

            topping_str = ', '.join(toppings) if toppings else 'classic toppings'
            lines.append(f'  Pizza {i}: {name.title()} ({size}) - {topping_str} -- ${price:.2f}')

        lines.append(f'  Estimated total: ${total:.2f}')
    else:
        lines.append('  Pizzas: None added yet.')

    lines.append('')
    if order['delivery_address']:
        lines.append(f'  Delivery address: {order["delivery_address"]}')
    else:
        lines.append('  Delivery address: Not provided yet.')

    if order['payment']:
        card = order['payment'].get('card_number', '')
        masked = '**** **** **** ' + card[-4:] if len(card) >= 4 else '****'
        lines.append(f'  Payment: Card ending in {masked}')
        if order['payment'].get('expiry'):
            lines.append(f'  Expiry: {order["payment"]["expiry"]}')
        if order['payment'].get('name_on_card'):
            lines.append(f'  Name on card: {order["payment"]["name_on_card"]}')
    else:
        lines.append('  Payment: Not provided yet.')

    return '\n'.join(lines)


# ──────────────────────────────────────────────────────────────────────
# Helper: build the menu display
# ──────────────────────────────────────────────────────────────────────

def _format_menu():
    """Return a nicely formatted menu string."""
    lines = []
    lines.append('Here is our menu! We have some wonderful specialty pizzas:')
    lines.append('')
    for name, toppings in PIZZA_MENU['specialty_pizzas'].items():
        lines.append(f'  {name.title():16s} - {", ".join(toppings)}')
    lines.append('')
    lines.append('Sizes and prices:')
    for size in PIZZA_MENU['sizes']:
        lines.append(f'  {size.title():8s}  ${PIZZA_MENU["prices"][size]:.2f}')
    lines.append(f'  (extra toppings beyond 3: +${PIZZA_MENU["topping_surcharge"]:.2f} each)')
    lines.append('')
    lines.append('You can also build your own pizza with any of these toppings:')
    lines.append(f'  {", ".join(PIZZA_MENU["available_toppings"])}')
    lines.append('')
    lines.append('Just tell me the name of a specialty pizza, or say "custom" to build your own!')
    return '\n'.join(lines)


# ──────────────────────────────────────────────────────────────────────
# Intent Detection
# ──────────────────────────────────────────────────────────────────────

def _detect_intent(text):
    """
    Analyze user input and return an intent string.
    Returns None if the input does not relate to pizza ordering.
    """
    t = text.lower().strip()

    # Summary / status request
    summary_phrases = [
        'what do you know', 'order so far', 'my order', 'order summary',
        'order status', 'what have i', 'show order', 'tell me what you know',
        'everything you know', 'what is known', 'review order',
    ]
    if any(phrase in t for phrase in summary_phrases):
        return 'summary'

    # Cancel
    cancel_phrases = ['cancel order', 'cancel my order', 'forget it', 'never mind', 'start over']
    if any(phrase in t for phrase in cancel_phrases):
        return 'cancel'

    # Menu request
    menu_phrases = ['menu', 'what do you have', 'what pizzas', 'show me options', 'what can i order', 'options']
    if any(phrase in t for phrase in menu_phrases):
        return 'show_menu'

    # Detect specialty pizza names
    for pizza_name in PIZZA_MENU['specialty_pizzas']:
        if pizza_name in t:
            return f'named_pizza:{pizza_name}'

    # Custom pizza trigger
    if 'custom' in t or 'build my own' in t or 'build your own' in t or 'make my own' in t:
        return 'custom_pizza'

    # Size detection
    for size in PIZZA_MENU['sizes']:
        if size in t:
            return f'size:{size}'

    # Topping detection
    found_toppings = [tp for tp in PIZZA_MENU['available_toppings'] if tp in t]
    if found_toppings:
        return 'toppings:' + ','.join(found_toppings)

    # Done ordering pizzas
    done_phrases = [
        "that's all", 'that is all', 'done ordering', 'no more',
        'nothing else', 'checkout', 'place order', 'finish order',
        'complete order', 'just that', 'no thanks', 'nope',
    ]
    if any(phrase in t for phrase in done_phrases):
        return 'done_ordering'

    # Yes / affirmative
    if t in ['yes', 'yeah', 'yep', 'sure', 'ok', 'okay', 'absolutely', 'yes please', 'yup', 'y']:
        return 'yes'

    # No / negative
    if t in ['no', 'nah', 'nope', 'n', 'no thanks']:
        return 'no'

    # Order initiation (kept near the end so it does not mask
    # more specific intents like named pizzas or toppings)
    order_phrases = [
        'order a pizza', 'order pizza', 'want a pizza', 'want pizza',
        'get a pizza', 'get pizza', 'like a pizza', 'like pizza',
        'buy a pizza', 'buy pizza', 'i want to order', 'place an order',
        'can i order', 'hungry',
    ]
    if any(phrase in t for phrase in order_phrases):
        return 'start_order'

    if ('i would like' in t or "i'd like" in t) and ('pizza' in t or 'order' in t):
        return 'start_order'

    return None


# ──────────────────────────────────────────────────────────────────────
# The Custom Logic Adapter
# ──────────────────────────────────────────────────────────────────────

class PizzaOrderLogicAdapter(LogicAdapter):
    """
    Custom ChatterBot logic adapter that manages a full pizza ordering
    conversation, including pizza selection, delivery address, and
    credit card payment collection.
    """

    def __init__(self, chatbot, **kwargs):
        super().__init__(chatbot, **kwargs)

    def can_process(self, statement):
        """
        Return True if this adapter should handle the input.
        We handle it when a pizza related intent is detected
        or an order is already in progress.
        """
        intent = _detect_intent(statement.text)
        if order['stage'] != 'idle':
            return True
        if intent is not None:
            return True
        return False

    def process(self, input_statement, additional_response_selection_parameters):
        """
        Process the input and return an appropriate response
        based on the current order stage and detected intent.
        """
        global order
        text = input_statement.text
        intent = _detect_intent(text)
        stage = order['stage']
        response_text = ''
        confidence = 0.95

        # ──── SUMMARY (available at any stage) ────
        if intent == 'summary':
            response_text = _format_summary()

        # ──── CANCEL (available at any stage) ─────
        elif intent == 'cancel':
            reset_order()
            response_text = (
                "No worries at all! I have cleared your order. "
                "Whenever you are ready to start fresh, just let me know!"
            )

        # ──── SHOW MENU (available at any stage) ──
        elif intent == 'show_menu':
            response_text = _format_menu()

        # ──── IDLE STAGE ──────────────────────────
        elif stage == 'idle':
            if intent == 'start_order':
                order['stage'] = 'ordering'
                response_text = (
                    "Oh wonderful, I would love to help you with that! "
                    "Would you like to pick from our specialty pizzas, or build your own? "
                    "You can also say 'menu' to see all our options."
                )
            elif intent and intent.startswith('named_pizza:'):
                pizza_name = intent.split(':', 1)[1]
                order['stage'] = 'awaiting_size'
                order['current_pizza'] = {
                    'name': pizza_name,
                    'toppings': list(PIZZA_MENU['specialty_pizzas'][pizza_name]),
                }
                toppings_str = ', '.join(PIZZA_MENU['specialty_pizzas'][pizza_name])
                sp = PIZZA_MENU['prices']['small']
                mp = PIZZA_MENU['prices']['medium']
                lp = PIZZA_MENU['prices']['large']
                response_text = (
                    f"Great taste! A {pizza_name.title()} it is, that comes with {toppings_str}. "
                    f"What size would you like? We have small (${sp:.2f}), "
                    f"medium (${mp:.2f}), or large (${lp:.2f})."
                )
            else:
                confidence = 0.65
                response_text = (
                    "Hey there! I am your pizza ordering assistant. "
                    "Just say 'I want to order a pizza' to get started, "
                    "or 'menu' to see what we have!"
                )

        # ──── ORDERING STAGE ──────────────────────
        elif stage == 'ordering':
            if intent and intent.startswith('named_pizza:'):
                pizza_name = intent.split(':', 1)[1]
                order['stage'] = 'awaiting_size'
                order['current_pizza'] = {
                    'name': pizza_name,
                    'toppings': list(PIZZA_MENU['specialty_pizzas'][pizza_name]),
                }
                toppings_str = ', '.join(PIZZA_MENU['specialty_pizzas'][pizza_name])
                response_text = (
                    f"Lovely choice! The {pizza_name.title()} comes with {toppings_str}. "
                    f"What size would you like? Small, medium, or large?"
                )
            elif intent == 'custom_pizza':
                order['stage'] = 'awaiting_toppings'
                order['current_pizza'] = {'name': 'Custom pizza', 'toppings': []}
                available = ', '.join(PIZZA_MENU['available_toppings'])
                response_text = (
                    f"Awesome, let us build your perfect pizza! "
                    f"Which toppings would you like? Here is what we have: {available}. "
                    f"Feel free to list as many as you would like!"
                )
            elif intent and intent.startswith('toppings:'):
                topping_list = intent.split(':', 1)[1].split(',')
                order['stage'] = 'awaiting_size'
                order['current_pizza'] = {'name': 'Custom pizza', 'toppings': topping_list}
                topping_display = ', '.join(topping_list)
                response_text = (
                    f"Nice picks! I have added {topping_display} to your custom pizza. "
                    f"What size would you like? Small, medium, or large?"
                )
            elif intent and intent.startswith('size:'):
                response_text = (
                    "I have noted the size! But first, which pizza would you like? "
                    "You can pick a specialty like Margherita or Pepperoni, or say 'custom' to build your own."
                )
            else:
                response_text = (
                    "Sure thing! Would you like one of our specialty pizzas "
                    "(like Margherita, Pepperoni, Hawaiian, Veggie, Meat Lovers, BBQ Chicken, or Supreme), "
                    "or would you prefer to build a custom pizza? "
                    "Just say the pizza name or 'custom' to get started!"
                )

        # ──── AWAITING TOPPINGS ───────────────────
        elif stage == 'awaiting_toppings':
            if intent and intent.startswith('toppings:'):
                topping_list = intent.split(':', 1)[1].split(',')
                order['current_pizza']['toppings'].extend(topping_list)
                order['stage'] = 'awaiting_size'
                all_toppings = ', '.join(order['current_pizza']['toppings'])
                response_text = (
                    f"Sounds delicious! Your pizza now has: {all_toppings}. "
                    f"What size would you like? Small, medium, or large?"
                )
            elif intent == 'done_ordering' or intent == 'no':
                if order['current_pizza'].get('toppings'):
                    order['stage'] = 'awaiting_size'
                    response_text = (
                        "Perfect! What size would you like for this pizza? "
                        "Small, medium, or large?"
                    )
                else:
                    response_text = (
                        "Hmm, it looks like we have not added any toppings yet. "
                        "Could you let me know which toppings you would like?"
                    )
            else:
                available = ', '.join(PIZZA_MENU['available_toppings'])
                response_text = (
                    f"I would love to add those toppings! Could you pick from our list? "
                    f"We have: {available}"
                )

        # ──── AWAITING SIZE ───────────────────────
        elif stage == 'awaiting_size':
            if intent and intent.startswith('size:'):
                size = intent.split(':', 1)[1]
                order['current_pizza']['size'] = size
                order['items'].append(dict(order['current_pizza']))
                order['current_pizza'] = {}
                order['stage'] = 'awaiting_more'

                pizza_name = order['items'][-1]['name']
                count = len(order['items'])
                s_suffix = 's' if count > 1 else ''
                response_text = (
                    f"Wonderful! I have added a {size} {pizza_name.title()} to your order. "
                    f"That is {count} pizza{s_suffix} so far. "
                    f"Would you like to add another pizza, or are you all set?"
                )
            else:
                sp = PIZZA_MENU['prices']['small']
                mp = PIZZA_MENU['prices']['medium']
                lp = PIZZA_MENU['prices']['large']
                response_text = (
                    "I just need the size for your pizza. "
                    f"We have small (${sp:.2f}), "
                    f"medium (${mp:.2f}), or "
                    f"large (${lp:.2f}). Which would you prefer?"
                )

        # ──── AWAITING MORE PIZZAS ────────────────
        elif stage == 'awaiting_more':
            if intent == 'yes' or intent == 'start_order':
                order['stage'] = 'ordering'
                response_text = (
                    "Great, let us add another one! Which pizza would you like next? "
                    "Specialty name or 'custom' to build your own."
                )
            elif intent and intent.startswith('named_pizza:'):
                pizza_name = intent.split(':', 1)[1]
                order['stage'] = 'awaiting_size'
                order['current_pizza'] = {
                    'name': pizza_name,
                    'toppings': list(PIZZA_MENU['specialty_pizzas'][pizza_name]),
                }
                response_text = (
                    f"Ooh nice, adding a {pizza_name.title()}! What size for this one?"
                )
            elif intent in ('no', 'done_ordering') or intent is None:
                order['stage'] = 'awaiting_address'
                response_text = (
                    "Alright, your pizzas are all set! "
                    "Now, could you share the delivery address where we should send your order?"
                )
            else:
                order['stage'] = 'awaiting_address'
                response_text = (
                    "Sounds like you are all set with pizzas! "
                    "Could you please share your delivery address?"
                )

        # ──── AWAITING ADDRESS ────────────────────
        elif stage == 'awaiting_address':
            if len(text.strip()) > 3:
                order['delivery_address'] = text.strip()
                order['stage'] = 'awaiting_payment'
                response_text = (
                    f"Got it, delivering to: {text.strip()}. "
                    f"Almost done! Could you please provide your credit card details? "
                    f"I will need the card number, expiration date, and the name on the card. "
                    f"You can share them all at once, for example: "
                    f"'4111 1111 1111 1234, 12/27, John Doe'"
                )
            else:
                response_text = (
                    "I would need a bit more detail for the delivery address. "
                    "Could you provide the full street address, please?"
                )

        # ──── AWAITING PAYMENT ────────────────────
        elif stage == 'awaiting_payment':
            import re
            parts = [p.strip() for p in text.split(',')]
            card_number = ''
            expiry = ''
            name_on_card = ''

            for part in parts:
                digits_only = re.sub(r'\D', '', part)
                if len(digits_only) >= 13 and len(digits_only) <= 19:
                    card_number = digits_only
                elif re.match(r'^\d{1,2}[/\-]\d{2,4}$', part.strip()):
                    expiry = part.strip()
                elif re.match(r'^[A-Za-z\s]{2,}$', part.strip()):
                    name_on_card = part.strip()

            if card_number:
                order['payment'] = {
                    'card_number': card_number,
                    'expiry': expiry if expiry else 'not provided',
                    'name_on_card': name_on_card if name_on_card else 'not provided',
                }
                order['stage'] = 'confirmed'
                masked = '**** **** **** ' + card_number[-4:]

                summary = _format_summary()
                response_text = (
                    f"Thank you so much! Payment done (card ending in {masked}).\n\n"
                    f"{summary}\n\n"
                    f"Your order has been placed! Thank you for choosing Pizza Palace. "
                    f"Your delicious pizza is on its way. Enjoy your meal!"
                )
            else:
                response_text = (
                    "I was not quite able to read the card details. No worries! "
                    "Could you share them in this format: "
                    "'card number, expiry (MM/YY), name on card'?\n"
                    "For example: '4111 1111 1111 1234, 12/27, John Doe'"
                )

        # ──── CONFIRMED STAGE ─────────────────────
        elif stage == 'confirmed':
            if intent == 'start_order':
                reset_order()
                order['stage'] = 'ordering'
                response_text = (
                    "A fresh new order! What pizza can I get for you this time?"
                )
            elif intent == 'summary':
                response_text = _format_summary()
            else:
                response_text = (
                    "Your order is already placed and on its way! "
                    "If you would like to place a new order, just say 'order pizza'. "
                    "Thanks again for ordering with us!"
                )

        # ──── FALLBACK ────────────────────────────
        else:
            confidence = 0.4
            response_text = (
                "I am here to help with your pizza order! "
                "Say 'order pizza' to get started, or 'menu' to see our options."
            )

        response_statement = Statement(text=response_text)
        response_statement.confidence = confidence
        return response_statement

Overwriting pizza_adapter.py


## 3. Chatbot Initialization & Training

We set up the ChatBot with two logic adapters and **SQLite in-memory storage** (no database file):
1. **PizzaOrderLogicAdapter** (custom) - handles the pizza ordering workflow with high confidence
2. **BestMatch** (built in) - handles general chitchat using ListTrainer data

In [36]:
import importlib
import sys
from chatterbot import ChatBot
from chatterbot.trainers import ListTrainer

# Reload adapter module so notebook reruns always pick up latest logic changes
if 'pizza_adapter' in sys.modules:
    importlib.reload(sys.modules['pizza_adapter'])

# Create the chatbot with our custom adapter + BestMatch for general chat
bot = ChatBot(
    'PizzaPal',
    storage_adapter='chatterbot.storage.SQLStorageAdapter',
    database_uri='sqlite:///:memory:',
    logic_adapters=[
        {
            'import_path': 'pizza_adapter.PizzaOrderLogicAdapter',
        },
        {
            'import_path': 'chatterbot.logic.BestMatch',
            'default_response': (
                "I appreciate you chatting with me! I am best at helping with pizza orders though. "
                "Just say 'order pizza' or 'menu' whenever you are ready!"
            ),
            'maximum_similarity_threshold': 0.90,
        },
    ],
)

print('PizzaPal chatbot created successfully!')

PizzaPal chatbot created successfully!


In [37]:
# ──────────────────────────────────────────────────────────────────────
# Training Data: General conversation + pizza context
# ──────────────────────────────────────────────────────────────────────

trainer = ListTrainer(bot)

# Greetings and casual conversation
trainer.train([
    'Hello',
    'Hey there! Welcome to Pizza Palace. How can I help you today?',
])
trainer.train([
    'Hi',
    'Hi! Welcome to Pizza Palace. Would you like to order a pizza?',
])
trainer.train([
    'Good morning',
    'Good morning! Great to see you. Ready for some delicious pizza?',
])
trainer.train([
    'Good evening',
    'Good evening! Perfect time for pizza. What can I get for you?',
])
trainer.train([
    'Hey',
    'Hey! Welcome to Pizza Palace. Feeling hungry?',
])

# General questions about the bot
trainer.train([
    'What can you do?',
    'I can help you order delicious pizzas! We have specialty pizzas and custom options. Just say "order pizza" or "menu" to get started.',
])
trainer.train([
    'Who are you?',
    'I am PizzaPal, your friendly pizza ordering assistant at Pizza Palace! I can help you pick the perfect pizza, take your delivery address, and handle payment.',
])
trainer.train([
    'How are you?',
    'I am doing great, thank you for asking! Ready to make your day better with some amazing pizza.',
])

# Thank you / goodbye
trainer.train([
    'Thank you',
    'You are very welcome! Enjoy your meal and have a wonderful day!',
])
trainer.train([
    'Thanks',
    'Happy to help! Enjoy your pizza!',
])
trainer.train([
    'Goodbye',
    'Goodbye! Hope to see you again soon. Have a wonderful day!',
])
trainer.train([
    'Bye',
    'Bye for now! Come back anytime you are craving pizza!',
])

# Pizza related general questions
trainer.train([
    'What is your most popular pizza?',
    'Our Pepperoni and Margherita are fan favorites! The Meat Lovers is also super popular. Say "menu" to see all our options!',
])
trainer.train([
    'Do you have vegetarian options?',
    'Absolutely! Our Veggie pizza is loaded with bell peppers, mushrooms, onions, and olives. You can also build a custom pizza with any veggie toppings you like!',
])
trainer.train([
    'How long does delivery take?',
    'Typically 30 to 45 minutes, depending on your location. We always try to get your pizza to you as fresh as possible!',
])
trainer.train([
    'Do you deliver?',
    'Yes, we deliver! Just place your order and share your address, and we will bring the pizza right to your door.',
])

print('Training complete! PizzaPal is ready to take orders.')

List Trainer: 100%|██████████| 2/2 [00:00<00:00, 35246.25it/s]

Training complete! PizzaPal is ready to take orders.


## 4. Helper Functions for Interaction

In [44]:
from pizza_adapter import reset_order


def chat(user_input):
    """
    Send a message to PizzaPal and print the conversation.
    Used for the scripted demo below.
    """
    print(f'You:      {user_input}')
    response = bot.get_response(user_input)
    print(f'PizzaPal: {response}')
    print()


def interactive_chat():
    """
    Start an interactive chat session with PizzaPal.
    Type 'quit' or 'exit' to end the session.
    """
    print('=' * 60)
    print('  Welcome to Pizza Palace!')
    print('  Type your message below. Type "quit" to exit.')
    print('=' * 60)
    print()

    reset_order()

    while True:
        try:
            user_input = input('You:      ')
            print(f'You:      {user_input}')
            if user_input.lower().strip() in ('quit', 'exit', 'q'):
                print('PizzaPal: Thanks for visiting Pizza Palace! See you next time!')
                break
            response = bot.get_response(user_input)
            print(f'PizzaPal: {response}')
            print()
        except (KeyboardInterrupt, EOFError):
            print('\nPizzaPal: Goodbye! Come back anytime!')
            break


print('Chat functions ready.')

Chat functions ready.


## 5. Sample Run: Full Feature Demonstration

This scripted demo walks through the **complete ordering workflow**, showcasing every required feature:

1. **General greeting** (handled by BestMatch)
2. **Viewing the menu**
3. **Ordering a specialty pizza** (Pepperoni) with size selection
4. **Adding a second pizza** with a different specialty (Hawaiian)
5. **Requesting order summary** ("what do you know so far")
6. **Providing delivery address**
7. **Providing credit card payment details**
8. **Final order confirmation with full summary**

In [45]:
# Reset order state for a clean demo
reset_order()

print('=' * 60)
print('  SAMPLE RUN: Pizza Ordering Chatbot Demo')
print('=' * 60)
print()

# 1. Greeting
chat('Hello')

# 2. View the menu
chat('Can you show me the menu?')

# 3. Start ordering - specialty pizza
chat('I would like to order a pepperoni pizza')

# 4. Select size
chat('Large please')

# 5. Add another pizza
chat('Yes, I would also like a hawaiian')

# 6. Select size for second pizza
chat('Medium')

# 7. Check what the bot knows so far
chat('What do you know so far?')

# 8. Done ordering pizzas
chat("That's all for the pizzas")

# 9. Provide delivery address
chat('123 Huntington Ave, Apt 4B, Boston MA 02115')

# 10. Provide credit card payment details
chat('4532 7891 2345 6789, 09/27, Ruthvik Nath')

print('=' * 60)
print('  END OF DEMO')
print('=' * 60)

  SAMPLE RUN: Pizza Ordering Chatbot Demo

You:      Hello
PizzaPal: Hey there! Welcome to Pizza Palace. How can I help you today?

You:      Can you show me the menu?
PizzaPal: Here is our menu! We have some wonderful specialty pizzas:

  Margherita       - mozzarella, tomato sauce, fresh basil
  Pepperoni        - mozzarella, pepperoni, tomato sauce
  Hawaiian         - mozzarella, ham, pineapple, tomato sauce
  Veggie           - mozzarella, bell peppers, mushrooms, onions, olives, tomato sauce
  Meat Lovers      - mozzarella, pepperoni, sausage, bacon, ham, tomato sauce
  Bbq Chicken      - mozzarella, grilled chicken, red onions, bbq sauce
  Supreme          - mozzarella, pepperoni, sausage, bell peppers, mushrooms, onions, olives

Sizes and prices:
  Small     $8.99
  Medium    $11.99
  Large     $14.99
  (extra toppings beyond 3: +$1.50 each)

You can also build your own pizza with any of these toppings:
  pepperoni, sausage, mushrooms, onions, olives, bell peppers, bacon, ham, 

## 6. Additional Demo: Custom Pizza & Order Summary

This demo shows building a **custom pizza** with individual toppings and using the **order summary** feature at various points.

In [46]:
# Reset for a second demo
reset_order()

print('=' * 60)
print('  SAMPLE RUN 2: Custom Pizza & Summary Flow')
print('=' * 60)
print()

# 1. Start with a direct order
chat('I want to order a pizza')

# 2. Build a custom pizza
chat('I want a custom pizza')

# 3. Add toppings
chat('I would like mushrooms, extra cheese, and bacon')

# 4. Pick size
chat('Small')

# 5. No more pizzas
chat('No thanks')

# 6. Provide address
chat('360 Huntington Ave, Boston MA 02115')

# 7. Provide payment
chat('5425 1234 5678 9012, 03/28, Ruthvik Bandari')

# 8. Final summary (shows pizza + address + payment details)
chat('Show me my order summary')

print('=' * 60)
print('  END OF DEMO 2')
print('=' * 60)

  SAMPLE RUN 2: Custom Pizza & Summary Flow

You:      I want to order a pizza
PizzaPal: Oh wonderful, I would love to help you with that! Would you like to pick from our specialty pizzas, or build your own? You can also say 'menu' to see all our options.

You:      I want a custom pizza
PizzaPal: Awesome, let us build your perfect pizza! Which toppings would you like? Here is what we have: pepperoni, sausage, mushrooms, onions, olives, bell peppers, bacon, ham, pineapple, jalapenos, extra cheese, tomatoes, grilled chicken, anchovies, fresh basil, spinach. Feel free to list as many as you would like!

You:      I would like mushrooms, extra cheese, and bacon
PizzaPal: Sounds delicious! Your pizza now has: mushrooms, bacon, extra cheese. What size would you like? Small, medium, or large?

You:      Small
PizzaPal: Wonderful! I have added a small Custom Pizza to your order. That is 1 pizza so far. Would you like to add another pizza, or are you all set?

You:      No thanks
PizzaPal: Alr

## 7. Interactive Chat (Optional)

Uncomment and run the cell below to start an interactive chat session with PizzaPal. Type `quit` to exit.

In [47]:
# Set to True when you want to start live interactive chat
RUN_INTERACTIVE_CHAT = True

if RUN_INTERACTIVE_CHAT:
    interactive_chat()
else:
    print('Interactive chat is ready. Set RUN_INTERACTIVE_CHAT = True to start.')

  Welcome to Pizza Palace!
  Type your message below. Type "quit" to exit.

You:      hi
PizzaPal: Hi! Welcome to Pizza Palace. Would you like to order a pizza?

You:      yes I want to order a pizza
PizzaPal: Oh wonderful, I would love to help you with that! Would you like to pick from our specialty pizzas, or build your own? You can also say 'menu' to see all our options.

You:      Build my own
PizzaPal: Awesome, let us build your perfect pizza! Which toppings would you like? Here is what we have: pepperoni, sausage, mushrooms, onions, olives, bell peppers, bacon, ham, pineapple, jalapenos, extra cheese, tomatoes, grilled chicken, anchovies, fresh basil, spinach. Feel free to list as many as you would like!

You:      Pineapple and onions
PizzaPal: Sounds delicious! Your pizza now has: onions, pineapple. What size would you like? Small, medium, or large?

You:      medium
PizzaPal: Wonderful! I have added a medium Custom Pizza to your order. That is 1 pizza so far. Would you like to